# Grayscale and DINOv2-FFT Co-Training

Interactive execution of the grayscale SqueezeNet and DINOv2/FFT/XGBoost co-training experiment. Run the setup and definition cells in order, then choose either one configurable experiment or a grid.

In [ ]:
import csv
import os
from datetime import datetime
from pathlib import Path

import mlflow
import mlflow.pytorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import models, transforms
from torchvision.models import SqueezeNet1_1_Weights

from Scripts.BlumMitchellCoTraining import BlumMitchellCoTraining
from Scripts.RGBWithFFTDataset import RGBWithFFTDataset
from Scripts.fft_ensemble import FFTEnsembleModel
from Scripts.helper_functions import serialize_confusion_matrix

PROJECT_ROOT = Path.cwd()
DATASET_BASE_PATH = Path("D:/Facultate/Disertatie/mainProject/pythonProject1")
MODEL_OUTPUT_PATH = PROJECT_ROOT / "models"
RESULT_OUTPUT_PATH = PROJECT_ROOT / "small_80_experiment_results.csv"
MODEL_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

## 2. Configure MLflow Experiment Tracking

In [ ]:
# Set this before running if your MLflow server is remote.
# mlflow.set_tracking_uri("http://localhost:5000")

EXPERIMENT_NAME = "Fetal_Plane_CoTraining"
mlflow.set_experiment(EXPERIMENT_NAME)
active_experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
print(f"MLflow experiment: {active_experiment.name}")
print(f"Artifact location: {active_experiment.artifact_location}")

## 3. Define Experiment Configuration

In [ ]:
class ExperimentConfig:
    def __init__(self, dataset_type, cotraining_start, conf_grayscale, conf_fft):
        self.dataset_type = dataset_type
        self.cotraining_start = cotraining_start
        self.conf_grayscale = conf_grayscale
        self.conf_fft = conf_fft
        self.input_size = (227, 227)
        self.batch_size = 30
        self.num_epochs = 60
        self.learning_rate = 1e-4
        self.k = 200
        self.checked_number = 100
        self.cotraining_batch_size = 50

        dataset_map = {
            "small_80": {"base": "small_labeled_ultrasound_dataset", "unlabeled_pct": 80},
            # "organized_50": {"base": "organized_labeled_ultrasound_dataset", "unlabeled_pct": 50},
            # "large_20": {"base": "large_labeled_ultrasound_dataset", "unlabeled_pct": 20},
        }
        dataset_info = dataset_map[dataset_type]
        dataset_root = DATASET_BASE_PATH / dataset_info["base"]
        self.labeled_path = dataset_root / "labeled_train"
        self.unlabeled_path = dataset_root / "unlabeled_train"
        self.val_path = dataset_root / "validation"
        self.test_path = dataset_root / "test"
        self.unlabeled_pct = dataset_info["unlabeled_pct"]
        self.experiment_id = (
            f"{dataset_type}_start{cotraining_start}_"
            f"grayscale{conf_grayscale}_fft{conf_fft}"
        )

## 4. Initialize the Grayscale SqueezeNet Model

In [ ]:
def initialize_ensemble_model(num_classes, device):
    model_grayscale = models.squeezenet1_1(weights=SqueezeNet1_1_Weights.IMAGENET1K_V1)
    grayscale_input_layer = nn.Conv2d(1, 64, kernel_size=3, stride=2)
    with torch.no_grad():
        grayscale_input_layer.weight.copy_(model_grayscale.features[0].weight.mean(dim=1, keepdim=True))
    model_grayscale.features[0] = grayscale_input_layer
    model_grayscale.classifier[1] = nn.Conv2d(
        model_grayscale.classifier[1].in_channels, num_classes, kernel_size=1
    )
    model_grayscale.num_classes = num_classes
    return model_grayscale.to(device)

## 5. Initialize the FFT Ensemble Model

In [ ]:
def initialize_fft_model(num_classes, device):
    return FFTEnsembleModel(num_classes, device)

## 6. Create Dataset DataLoaders

In [ ]:
def create_loaders(grayscale_data, fft_data, unlabeled_data, val_data, test_data, batch_size):
    grayscale_loader = DataLoader(grayscale_data, batch_size=batch_size, shuffle=True)
    fft_loader = DataLoader(fft_data, batch_size=batch_size, shuffle=True)
    unlabeled_loader = DataLoader(unlabeled_data, batch_size=batch_size, shuffle=False)
    val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)
    return grayscale_loader, fft_loader, unlabeled_loader, val_loader, test_loader

## 7. Prepare Image Transforms and Datasets

Set `config` here to inspect the selected dataset before model construction.

In [ ]:
config = ExperimentConfig(
    dataset_type="small_80",
    cotraining_start=5,
    conf_grayscale=0.95,
    conf_fft=0.90,
)

grayscale_transform = transforms.Compose([
    transforms.Resize(config.input_size),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize([0.485], [0.229]),
])
dino_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
fft_transform = transforms.Compose([
    transforms.Resize(config.input_size),
    transforms.Lambda(lambda tensor: tensor.unsqueeze(0) if tensor.dim() == 2 else tensor),
    transforms.Normalize([0.485], [0.229]),
])

grayscale_dataset = RGBWithFFTDataset(config.labeled_path, grayscale_transform, fft_transform, dino_transform, labeled=True)
fft_dataset = RGBWithFFTDataset(config.labeled_path, grayscale_transform, fft_transform, dino_transform, labeled=True)
unlabeled_dataset = RGBWithFFTDataset(config.unlabeled_path, grayscale_transform, fft_transform, dino_transform, labeled=False)
val_dataset = RGBWithFFTDataset(config.val_path, grayscale_transform, fft_transform, dino_transform, labeled=True)
test_dataset = RGBWithFFTDataset(config.test_path, grayscale_transform, fft_transform, dino_transform, labeled=True)

print(f"Classes: {grayscale_dataset.classes}")
print(f"Class count: {len(grayscale_dataset.classes)}")
print(f"Labeled: {len(grayscale_dataset)}, unlabeled: {len(unlabeled_dataset)}")
print(f"Validation: {len(val_dataset)}, test: {len(test_dataset)}")

## 8. Configure Device, Models, Optimizer, and Co-Training

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(grayscale_dataset.classes)
model_grayscale = initialize_ensemble_model(num_classes, device)
model_ensemble = initialize_fft_model(num_classes, device)

cotrainer = BlumMitchellCoTraining(
    model_grayscale, model_ensemble, num_classes, device,
    cotraining_start=config.cotraining_start,
    k=config.k,
    confidence_thresh_fft=config.conf_fft,
    confidence_thresh_grayscale=config.conf_grayscale,
    checked_number=config.checked_number,
)
cotrainer.set_datasets(grayscale_dataset, fft_dataset, unlabeled_dataset)
optimizer_grayscale = optim.Adam(model_grayscale.parameters(), lr=config.learning_rate)
cotrainer.init_schedulers(optimizer_grayscale, step_size=5, gamma=0.9)
print(f"Using device: {device}")

## 9. Run the Epoch-Level Co-Training Loop

Run this once after the preceding cells. It opens one MLflow run and keeps it active through final evaluation and persistence.

In [ ]:
mlflow_run = mlflow.start_run(run_name=config.experiment_id)
mlflow.log_params({
    "dataset_type": config.dataset_type,
    "cotraining_start": config.cotraining_start,
    "conf_grayscale": config.conf_grayscale,
    "conf_fft": config.conf_fft,
    "learning_rate": config.learning_rate,
    "num_epochs": config.num_epochs,
    "batch_size": config.batch_size,
    "k_samples": config.k,
})

for epoch in range(config.num_epochs):
    epoch_counter = epoch + 1
    grayscale_loader, fft_loader, unlabeled_loader, val_loader, _ = create_loaders(
        grayscale_dataset, fft_dataset, unlabeled_dataset, val_dataset, test_dataset, config.batch_size
    )
    reevaluate_flag = epoch_counter > config.cotraining_start and epoch_counter % 4 == 0
    cotrainer.train_iteration(
        grayscale_loader, fft_loader, unlabeled_loader, optimizer_grayscale, None,
        epoch_counter, config.cotraining_batch_size, reevaluate_flag,
    )
    grayscale_acc, fft_acc, combined_acc, _, _, _ = cotrainer.evaluate(val_loader)
    mlflow.log_metrics({
        "val_grayscale_acc": grayscale_acc,
        "val_fft_acc": fft_acc,
        "val_combined_acc": combined_acc,
        "unlabeled_samples_used": len(cotrainer.used_unlabeled_indices),
    }, step=epoch_counter)
    print(
        f"Epoch {epoch_counter}/{config.num_epochs} | "
        f"validation grayscale={grayscale_acc:.4f}, FFT={fft_acc:.4f}, combined={combined_acc:.4f} | "
        f"pseudo-labels={len(grayscale_dataset.pseudo_samples)}"
    )

## 10. Evaluate the Final Models on the Test Set

In [ ]:
_, _, _, _, test_loader = create_loaders(
    grayscale_dataset, fft_dataset, unlabeled_dataset, val_dataset, test_dataset, config.batch_size
)
(
    test_grayscale_acc,
    test_fft_acc,
    test_combined_acc,
    grayscale_cm,
    fft_cm,
    combined_cm,
) = cotrainer.evaluate(test_loader)

print(f"Test grayscale accuracy: {test_grayscale_acc:.4f}")
print(f"Test FFT ensemble accuracy: {test_fft_acc:.4f}")
print(f"Test combined accuracy: {test_combined_acc:.4f}")

## 11. Save Models, Metrics, Confusion Matrices, and Artifacts

In [ ]:
mlflow.log_metrics({
    "test_grayscale_acc": test_grayscale_acc,
    "test_fft_acc": test_fft_acc,
    "test_combined_acc": test_combined_acc,
})
mlflow.pytorch.log_model(model_grayscale, "model_grayscale")

model_grayscale_path = MODEL_OUTPUT_PATH / f"{config.experiment_id}_grayscale.pth"
model_ensemble_path = MODEL_OUTPUT_PATH / f"{config.experiment_id}_fft_ensemble"
torch.save(model_grayscale.state_dict(), model_grayscale_path)
model_ensemble.save(model_ensemble_path)
mlflow.log_artifact(str(model_ensemble_path.with_suffix(".pt")))
mlflow.log_artifact(str(model_ensemble_path.with_suffix(".json")))

gray_cm_string = serialize_confusion_matrix(grayscale_cm)
fft_cm_string = serialize_confusion_matrix(fft_cm)
combined_cm_string = serialize_confusion_matrix(combined_cm)
results = {
    "experiment_id": config.experiment_id,
    "dataset": config.dataset_type,
    "unlabeled_pct": config.unlabeled_pct,
    "cotraining_start": config.cotraining_start,
    "conf_grayscale": config.conf_grayscale,
    "conf_fft": config.conf_fft,
    "test_grayscale_acc": test_grayscale_acc,
    "test_fft_acc": test_fft_acc,
    "test_combined_acc": test_combined_acc,
    "grayscale_confusion_matrix": gray_cm_string,
    "fft_confusion_matrix": fft_cm_string,
    "combined_confusion_matrix": combined_cm_string,
    "num_classes": len(grayscale_cm),
    "final_grayscale_size": len(grayscale_dataset),
    "final_fft_size": len(fft_dataset),
    "unlabeled_used": len(cotrainer.used_unlabeled_indices),
    "grayscale_pseudo_samples": len(grayscale_dataset.pseudo_samples),
    "fft_pseudo_samples": len(fft_dataset.pseudo_samples),
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
}
mlflow.end_run()
results

## 12. Save Experiment Results to CSV

In [ ]:
def save_results_to_csv(results_list, filename=RESULT_OUTPUT_PATH):
    headers = [
        "experiment_id", "dataset", "unlabeled_pct", "cotraining_start",
        "conf_grayscale", "conf_fft", "test_grayscale_acc", "test_fft_acc",
        "test_combined_acc", "grayscale_confusion_matrix", "fft_confusion_matrix",
        "combined_confusion_matrix", "num_classes", "final_grayscale_size",
        "final_fft_size", "unlabeled_used", "grayscale_pseudo_samples",
        "fft_pseudo_samples", "timestamp",
    ]
    filename = Path(filename)
    file_exists = filename.is_file()
    with filename.open("a", newline="") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=headers)
        if not file_exists:
            writer.writeheader()
        writer.writerows(results_list)

# After Section 11, persist the displayed experiment result with:
# save_results_to_csv([results])

## 13. Run a Single Configurable Experiment

This helper runs the full workflow without relying on the interactive state from Sections 7-11.

In [ ]:
def run_experiment(config):
    grayscale_transform = transforms.Compose([
        transforms.Resize(config.input_size), transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(), transforms.Normalize([0.485], [0.229]),
    ])
    dino_transform = transforms.Compose([
        transforms.Resize((224, 224)), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    fft_transform = transforms.Compose([
        transforms.Resize(config.input_size),
        transforms.Lambda(lambda tensor: tensor.unsqueeze(0) if tensor.dim() == 2 else tensor),
        transforms.Normalize([0.485], [0.229]),
    ])
    grayscale_data = RGBWithFFTDataset(config.labeled_path, grayscale_transform, fft_transform, dino_transform, labeled=True)
    fft_data = RGBWithFFTDataset(config.labeled_path, grayscale_transform, fft_transform, dino_transform, labeled=True)
    unlabeled_data = RGBWithFFTDataset(config.unlabeled_path, grayscale_transform, fft_transform, dino_transform, labeled=False)
    val_data = RGBWithFFTDataset(config.val_path, grayscale_transform, fft_transform, dino_transform, labeled=True)
    test_data = RGBWithFFTDataset(config.test_path, grayscale_transform, fft_transform, dino_transform, labeled=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_grayscale = initialize_ensemble_model(len(grayscale_data.classes), device)
    model_ensemble = initialize_fft_model(len(grayscale_data.classes), device)
    cotrainer = BlumMitchellCoTraining(model_grayscale, model_ensemble, len(grayscale_data.classes), device,
        checked_number=config.checked_number, cotraining_start=config.cotraining_start, k=config.k,
        confidence_thresh_fft=config.conf_fft, confidence_thresh_grayscale=config.conf_grayscale)
    cotrainer.set_datasets(grayscale_data, fft_data, unlabeled_data)
    optimizer = optim.Adam(model_grayscale.parameters(), lr=config.learning_rate)
    cotrainer.init_schedulers(optimizer, step_size=5, gamma=0.9)

    with mlflow.start_run(run_name=config.experiment_id):
        mlflow.log_params({"dataset_type": config.dataset_type, "cotraining_start": config.cotraining_start,
                           "conf_grayscale": config.conf_grayscale, "conf_fft": config.conf_fft,
                           "learning_rate": config.learning_rate, "num_epochs": config.num_epochs,
                           "batch_size": config.batch_size, "k_samples": config.k})
        for epoch in range(config.num_epochs):
            loaders = create_loaders(grayscale_data, fft_data, unlabeled_data, val_data, test_data, config.batch_size)
            reevaluate = epoch + 1 > config.cotraining_start and (epoch + 1) % 4 == 0
            cotrainer.train_iteration(*loaders[:3], optimizer, None, epoch + 1, config.cotraining_batch_size, reevaluate)
            grayscale_acc, fft_acc, combined_acc, _, _, _ = cotrainer.evaluate(loaders[3])
            mlflow.log_metrics({"val_grayscale_acc": grayscale_acc, "val_fft_acc": fft_acc,
                                "val_combined_acc": combined_acc,
                                "unlabeled_samples_used": len(cotrainer.used_unlabeled_indices)}, step=epoch + 1)
        _, _, _, _, test_loader = create_loaders(grayscale_data, fft_data, unlabeled_data, val_data, test_data, config.batch_size)
        grayscale_acc, fft_acc, combined_acc, grayscale_cm, fft_cm, combined_cm = cotrainer.evaluate(test_loader)
        mlflow.log_metrics({"test_grayscale_acc": grayscale_acc, "test_fft_acc": fft_acc,
                            "test_combined_acc": combined_acc})
        mlflow.pytorch.log_model(model_grayscale, "model_grayscale")
        grayscale_path = MODEL_OUTPUT_PATH / f"{config.experiment_id}_grayscale.pth"
        ensemble_path = MODEL_OUTPUT_PATH / f"{config.experiment_id}_fft_ensemble"
        torch.save(model_grayscale.state_dict(), grayscale_path)
        model_ensemble.save(ensemble_path)
        mlflow.log_artifact(str(ensemble_path.with_suffix(".pt")))
        mlflow.log_artifact(str(ensemble_path.with_suffix(".json")))

    return {"experiment_id": config.experiment_id, "dataset": config.dataset_type,
            "unlabeled_pct": config.unlabeled_pct, "cotraining_start": config.cotraining_start,
            "conf_grayscale": config.conf_grayscale, "conf_fft": config.conf_fft,
            "test_grayscale_acc": grayscale_acc, "test_fft_acc": fft_acc,
            "test_combined_acc": combined_acc,
            "grayscale_confusion_matrix": serialize_confusion_matrix(grayscale_cm),
            "fft_confusion_matrix": serialize_confusion_matrix(fft_cm),
            "combined_confusion_matrix": serialize_confusion_matrix(combined_cm),
            "num_classes": len(grayscale_cm), "final_grayscale_size": len(grayscale_data),
            "final_fft_size": len(fft_data), "unlabeled_used": len(cotrainer.used_unlabeled_indices),
            "grayscale_pseudo_samples": len(grayscale_data.pseudo_samples),
            "fft_pseudo_samples": len(fft_data.pseudo_samples),
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")}


def run_single_experiment(dataset_type, cotraining_start, conf_grayscale, conf_fft):
    result = run_experiment(ExperimentConfig(dataset_type, cotraining_start, conf_grayscale, conf_fft))
    save_results_to_csv([result])
    return result

## 14. Run an Experiment Grid

Edit the lists below to enable the datasets, start epochs, and threshold combinations required for an experiment sweep.

In [ ]:
def run_all_experiments():
    datasets = ["small_80"]
    cotraining_starts = [5]
    threshold_configs = [
        {"grayscale": 0.95, "fft": 0.90},
        # {"grayscale": 0.90, "fft": 0.85},
        # {"grayscale": 0.85, "fft": 0.80},
    ]
    results = []
    total = len(datasets) * len(cotraining_starts) * len(threshold_configs)
    for experiment_number, (dataset_type, start_epoch, thresholds) in enumerate(
        (
            (dataset_type, start_epoch, thresholds)
            for dataset_type in datasets
            for start_epoch in cotraining_starts
            for thresholds in threshold_configs
        ),
        start=1,
    ):
        print(f"Experiment {experiment_number}/{total}: {dataset_type}")
        try:
            result = run_single_experiment(
                dataset_type, start_epoch, thresholds["grayscale"], thresholds["fft"]
            )
            results.append(result)
        except Exception:
            import traceback
            traceback.print_exc()
    print(f"Completed {len(results)}/{total} experiments.")
    return results

# Uncomment to run the configured grid.
# all_results = run_all_experiments()